[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/03_wordpiece_unigram_sentencepiece/03_proyecto_wordpiece_unigram_sentencepiece.ipynb)

# Proyecto 03 · Entrenar y comparar BPE / WordPiece / Unigram sobre Políglota

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 2 h |
| **GPU recomendada** | Ninguna (CPU; `FAST_DEV_RUN = True` en minutos) |
| **Prerrequisitos** | Lección del módulo 03 (y módulos 01–02) |

## Contexto de negocio
Eres ML engineer en **Políglota Labs**. El buscador semántico necesitará su propio tokenizer (para el índice sparse hoy y para el embedder propio después). Tu CTO quiere una **recomendación con datos**: ¿BPE, WordPiece o Unigram?
Entrenarás los tres con el **mismo pipeline** sobre el corpus Políglota (ES/EN/código), los compararás con 4 métricas más un test de recuperación BM25, exportarás un BPE byte-level a `tiktoken` y escribirás la decisión.

> **Conexión.** Viene de: módulo 02 (tu BPE) y módulo 00 (corpus + BM25). Va hacia: módulo 04 (diseño de vocabulario y paridad multilingüe) y módulo 05 (qué falla cuando el vocabulario no coincide con los datos del modelo).

## Dataset
Corpus Políglota vía `carga_corpus()` (oficial HF Wikipedia ES/EN + CodeSearchNet → espejo Europarl + stdlib de Python → sintético; avisa en pantalla). Partición 80/20 por documento y por idioma. El mini golden set (consulta = primera frase de un documento) sirve para la prueba de retrieval.

## Entregables y rúbrica
1. `entrena_tokenizer(tipo, vocab_size, textos)` para `bpe | wordpiece | unigram` con idéntica normalización y pre-tokenización.
2. `evalua(tok, nombre)` con fertilidad y bytes/token por idioma, `[UNK]` en el texto de estrés y F1 morfológica.
3. Un analizador BM25 por tokenizer y su nDCG@10 en el mini golden set.
4. Exportación de un BPE byte-level a `tiktoken` con igualdad exacta.
5. Una decisión escrita (≥ 300 caracteres) apoyada en la tabla.

La celda **Rúbrica** puntúa automáticamente. Los umbrales se calibraron con lo que **mide la solución de referencia** (corpus espejo, `FAST_DEV_RUN=True`, vocabulario 4 000) menos un margen; si usas datos oficiales (Wikipedia) o `FAST_DEV_RUN=False`, los criterios *relativos* son los fiables y los absolutos se relajan.

In [ ]:
!pip install -q tokenizers sentencepiece tiktoken datasets matplotlib seaborn scikit-learn

In [ ]:
import os, sys, re, math, json, random, time, unicodedata, collections, itertools, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")          # evita sobre-suscripción de hilos
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU, minutos). False: escala completa (Colab Pro)
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
COL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "rojo": "#D62728"}
try:
    import torch
    torch.set_num_threads(1); torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN)

In [ ]:
# ---- Utilidades de corpus Políglota (copia local; contrato: carga_corpus(fast_dev) -> dict) ----
import urllib.request, urllib.error, zipfile, io, inspect, ast

def hay_red(url: str, timeout: float = 6.0) -> bool:
    """Sonda rápida para no esperar minutos si el host está bloqueado (proxy corporativo, etc.)."""
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=timeout)
        return True
    except urllib.error.HTTPError:
        return True          # el servidor respondió (aunque sea 4xx): hay ruta de red
    except Exception:
        return False         # túnel bloqueado, DNS, timeout...

def _hf_wikipedia(lang: str, n: int, max_chars: int):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    out = []
    for ex in ds:
        if len(ex["text"]) > 400:
            out.append({"id": f"wiki_{lang}_{len(out)}", "lang": lang, "text": ex["text"][:max_chars]})
        if len(out) >= n:
            break
    return out

def _hf_codesearchnet(n: int):
    from datasets import load_dataset
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("claudios/code_search_net", {"name": "python"}),
                     ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw)
            out = []
            for ex in ds:
                code = ex.get("func_code_string") or ex.get("code") or ex.get("original_string")
                if code and 80 < len(code) < 3000:
                    out.append({"id": f"csn_{len(out)}", "lang": "code", "text": code})
                if len(out) >= n:
                    return out
        except Exception as e:
            print("   (CodeSearchNet vía", repo, "falló:", type(e).__name__, ")")
    raise RuntimeError("CodeSearchNet no disponible")

def _detok(s: str) -> str:
    s = re.sub(r"\s+([,.;:!?%)\]])", r"\1", s)
    s = re.sub(r"([(\[¿¡])\s+", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
def _espejo_nltk(n: int):
    """Espejo público: muestra de Europarl (ES/EN) del repositorio nltk_data en GitHub (CC/uso de investigación)."""
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/europarl_raw.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
    docs = []
    for lang, folder, ext in [("es", "spanish", ".es"), ("en", "english", ".en")]:
        lines = []
        for name in sorted(z.namelist()):
            if f"/{folder}/" in name and name.endswith(ext):
                lines += [_detok(l) for l in z.read(name).decode("utf-8", "ignore").splitlines() if len(l.split()) > 6]
        step = 8
        chunks = [" ".join(lines[i:i + step]) for i in range(0, len(lines), step)]
        random.Random(SEED).shuffle(chunks)
        docs += [{"id": f"ep_{lang}_{i}", "lang": lang, "text": c[:2000]} for i, c in enumerate(chunks[:n])]
    return docs

def _codigo_stdlib(n: int):
    """Código Python real de la biblioteca estándar (siempre disponible, sin red)."""
    import json, re as _re, argparse, textwrap, collections as _c, itertools as _i, functools, pathlib, csv, heapq, bisect
    import statistics, difflib, string, shutil, tempfile, fractions, random as _r, datetime, calendar, glob, fnmatch, logging
    out = []
    for mod in [json.decoder, json.encoder, _re, argparse, textwrap, functools, pathlib, csv, heapq, bisect, statistics, difflib,
                shutil, tempfile, fractions, datetime, calendar, glob, fnmatch, logging, _c, _r]:
        try:
            tree = ast.parse(inspect.getsource(mod)); src = inspect.getsource(mod).splitlines()
        except (OSError, TypeError, SyntaxError):
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef,)) and 4 <= (node.end_lineno - node.lineno) <= 40:
                code = "\n".join(src[node.lineno - 1: node.end_lineno])
                if 80 < len(code) < 3000:
                    out.append({"id": f"std_{len(out)}", "lang": "code", "text": code})
    random.Random(SEED).shuffle(out)
    return out[:n]

In [ ]:
_ES_S = ["el sistema", "la base de datos", "el servidor", "la aplicación", "el modelo", "la biblioteca", "el usuario", "la función", "el módulo", "la consulta"]
_ES_V = ["procesa", "almacena", "devuelve", "actualiza", "valida", "descarga", "ejecuta", "registra", "indexa", "compara"]
_ES_O = ["los documentos relevantes", "una lista de resultados", "los parámetros de configuración", "las peticiones entrantes", "la información almacenada",
         "varias tablas de forma eficiente", "los errores más frecuentes", "las versiones anteriores", "los textos en diferentes idiomas", "un conjunto de datos nuevo"]
_ES_A = ["rápidamente", "automáticamente", "correctamente", "de manera incremental", "sin interrupciones", "cada vez que cambia la configuración", "antes de la instalación", "durante la inicialización"]
_EN_S = ["the system", "the database", "the server", "the application", "the model", "the library", "the user", "the function", "the module", "the query"]
_EN_V = ["processes", "stores", "returns", "updates", "validates", "downloads", "executes", "logs", "indexes", "compares"]
_EN_O = ["the relevant documents", "a list of results", "the configuration parameters", "the incoming requests", "the stored information",
         "several tables efficiently", "the most frequent errors", "the previous versions", "texts in different languages", "a new dataset"]
_EN_A = ["quickly", "automatically", "correctly", "incrementally", "without interruptions", "whenever the configuration changes", "before the installation", "during initialization"]

def _sintetico(n: int):
    """Último recurso: texto ES/EN generado por plantillas + funciones Python plantilla (mismo esquema)."""
    rng = random.Random(SEED); docs = []
    for lang, S, V, O, A in [("es", _ES_S, _ES_V, _ES_O, _ES_A), ("en", _EN_S, _EN_V, _EN_O, _EN_A)]:
        for i in range(n):
            sents = []
            for _ in range(rng.randint(4, 8)):
                s = f"{rng.choice(S)} {rng.choice(V)} {rng.choice(O)} {rng.choice(A)}."
                sents.append(s[0].upper() + s[1:])
            docs.append({"id": f"syn_{lang}_{i}", "lang": lang, "text": " ".join(sents)})
    names = ["load", "parse", "merge", "filter", "count", "split", "encode", "decode", "score", "rank"]
    things = ["items", "tokens", "rows", "words", "records", "paths", "values", "pairs"]
    for i in range(n):
        f, t = rng.choice(names), rng.choice(things)
        docs.append({"id": f"syn_code_{i}", "lang": "code", "text":
            f"def {f}_{t}({t}, limit={rng.randint(2, 99)}):\n    \"\"\"{f.capitalize()} the {t} and return at most limit results.\"\"\"\n"
            f"    result = []\n    for x in {t}:\n        if len(result) >= limit:\n            break\n        result.append(x)\n    return result\n"})
    return docs

In [ ]:
def _mini_golden(docs, n_q: int = 24):
    """Mini golden set: la consulta es la primera frase (≤ 12 palabras) del documento; qrels {qid: {doc_id: 1}}."""
    rng = random.Random(SEED); queries, qrels = [], {}
    for lang in ["es", "en", "code"]:
        pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool)
        for d in pool[: n_q // 3]:
            if lang == "code":
                try:
                    q = (ast.get_docstring(ast.parse(d["text"]).body[0]) or "").split("\n")[0]
                except SyntaxError:
                    q = ""
                q = q or re.sub(r"[_(].*", "", d["text"].split("def ")[-1]).replace("_", " ")
            else:
                q = " ".join(re.split(r"(?<=[.!?])\s", d["text"])[0].split()[:12])
            qid = f"q{len(queries)}"; queries.append({"qid": qid, "lang": lang, "text": q}); qrels[qid] = {d["id"]: 1}
    return queries, qrels

In [ ]:
def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota. Cadena: oficial (HF Wikipedia + CodeSearchNet) -> espejo (Europarl vía GitHub + stdlib) -> sintético.
    Devuelve {"docs", "queries", "qrels", "source": "real"|"synthetic", "detail": {...}}."""
    n = 150 if fast_dev else 3000
    max_chars = 2000 if fast_dev else 6000
    docs, detail = [], {}
    hf_ok = hay_red("https://huggingface.co")
    for lang in ["es", "en"]:
        got = None
        if hf_ok:
            try: got, detail[lang] = _hf_wikipedia(lang, n, max_chars), "hf:wikimedia/wikipedia"
            except Exception as e: print(f"   (Wikipedia {lang} falló: {type(e).__name__})")
        if got is None and hay_red("https://raw.githubusercontent.com"):
            try:
                mirror = _espejo_nltk(n); got = [d for d in mirror if d["lang"] == lang]; detail[lang] = "espejo:nltk-europarl"
            except Exception as e: print(f"   (espejo {lang} falló: {type(e).__name__})")
        if got is None: got, detail[lang] = [d for d in _sintetico(n) if d["lang"] == lang], "sintetico"
        docs += got
    code = None
    if hf_ok:
        try: code, detail["code"] = _hf_codesearchnet(n), "hf:code_search_net"
        except Exception as e: print(f"   (CodeSearchNet falló: {type(e).__name__})")
    if code is None:
        code = _codigo_stdlib(n); detail["code"] = "espejo:stdlib-python"
        if len(code) < 10: code, detail["code"] = [d for d in _sintetico(n) if d["lang"] == "code"], "sintetico"
    docs += code
    queries, qrels = _mini_golden(docs)
    real = all(not v.startswith("sintetico") for v in detail.values())
    if not real: print("AVISO: parte del corpus es SINTÉTICA (sin acceso a los datasets). Las cifras absolutas cambiarán con datos reales.")
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real" if real else "synthetic", "detail": detail}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs = corpus["docs"]
print("fuente:", corpus["source"], corpus["detail"])
by_lang = {l: [d["text"] for d in docs if d["lang"] == l] for l in ["es", "en", "code"]}
for l, ts in by_lang.items():
    print(f"{l:>4}: {len(ts):4d} docs | {sum(len(t) for t in ts)/1e3:8.1f} k caracteres | {sum(len(t.encode()) for t in ts)/1e3:8.1f} k bytes")
print("\nEjemplo ES:", by_lang["es"][0][:160].replace("\n", " "))
print("Ejemplo código:\n", by_lang["code"][0][:200])

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out
WORD_RE = re.compile(r"\w+(?:['’]\w+)?|[^\w\s]", re.UNICODE)
def palabras(text: str): return WORD_RE.findall(text)

def metricas_tokenizacion(encode, textos) -> dict:
    """encode: str -> list[str|int]. Devuelve fertilidad (tokens/palabra), bytes/token y caracteres/token."""
    n_tok = n_pal = n_byte = n_chr = 0
    for t in textos:
        n_tok += len(encode(t)); n_pal += len(palabras(t)); n_byte += len(t.encode("utf-8")); n_chr += len(t)
    return {"fertilidad": n_tok / max(n_pal, 1), "bytes_por_token": n_byte / max(n_tok, 1), "chars_por_token": n_chr / max(n_tok, 1), "tokens": n_tok}

In [ ]:
from collections import Counter
from tokenizers import Tokenizer, models, pre_tokenizers, normalizers, trainers, decoders, Regex
from tokenizers.pre_tokenizers import ByteLevel
import pandas as pd

rng = random.Random(SEED); train_docs, test_docs = [], []
for lang in ["es", "en", "code"]:
    pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool); cut = int(0.8 * len(pool))
    train_docs += pool[:cut]; test_docs += pool[cut:]
train_texts = [d["text"] for d in train_docs]
test_by_lang = {l: [d["text"] for d in test_docs if d["lang"] == l] for l in ["es", "en", "code"]}
VOCAB = 4000 if FAST_DEV_RUN else 16000
ESTRES = ["Привет, мир", "你好，世界", "مرحبا بالعالم", "नमस्ते दुनिया", "😀🚀 ¡hola!", "naïve café Ünïcödé"]
def n_unk(tok, texto): return tok.encode(texto).ids.count(tok.token_to_id("[UNK]"))   # cuenta por ID (HF Unigram devuelve el texto original)

MORF_ES = ["re|construc|ción", "des|confian|za", "im|pens|able", "trabaj|ador|es", "feliz|mente", "organiza|ciones", "publica|ción", "il|legal|mente",
           "descubri|miento", "pensa|miento", "agrupa|ción", "comunica|ción", "in|felic|idad", "des|hacer", "pre|histór|ico", "sub|desarrollo",
           "co|opera|ción", "automática|mente", "ir|reversible", "re|ubic|ar", "des|organiza|ción", "hermosa|mente", "re|comenda|ciones",
           "inter|nacional|es", "actualiza|ción", "indepen|diente|mente", "multi|lingüe", "pos|guerra", "sobre|ajuste", "ejecuta|ble"]
MORF_EN = ["un|believ|able", "re|construct|ion", "inter|national|iz|ation", "happi|ness", "un|happi|ly", "dis|agree|ment", "mis|understand|ing",
           "pre|process|ing", "re|us|able", "nation|al|ism", "effect|ive|ly", "token|iz|ation", "re|writ|ing", "non|linear", "sub|word",
           "multi|lingual", "over|fit|ting", "under|estimate", "counter|example", "teach|er|s", "re|commend|ation|s", "quick|ly",
           "in|depend|ent|ly", "un|control|led", "de|code", "en|code", "pre|train|ing", "fine|tun|ing", "dis|connect|ed", "work|ing|s"]
def fronteras(partes):
    pos, acc = set(), 0
    for p in partes[:-1]: acc += len(p); pos.add(acc)
    return pos
def f1_morfologico(segmenta, gold):
    tp = fp = fn = 0
    for g in gold:
        gp = g.split("|"); palabra = "".join(gp)
        pred = [p for p in (x.replace("▁", "").replace("##", "") for x in segmenta(palabra)) if p]
        if "".join(pred) != palabra: pred = [palabra]
        a, b = fronteras(pred), fronteras(gp); tp += len(a & b); fp += len(a - b); fn += len(b - a)
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return 2 * p * r / max(p + r, 1e-9)

# BM25 mínimo (módulo 00): lo recibes hecho; tu trabajo es el ANALIZADOR (el tokenizer)
class BM25:
    def __init__(self, textos, ids, analizador, k1=1.5, b=0.75):
        self.ids, self.k1, self.b = ids, k1, b
        self.tf = [Counter(analizador(t)) for t in textos]; self.len = np.array([sum(c.values()) for c in self.tf], float); self.avg = self.len.mean()
        df = Counter(w for c in self.tf for w in c); n = len(textos)
        self.idf = {w: math.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}
    def ranking(self, consulta_tokens, k=10):
        sc = np.zeros(len(self.tf))
        for w in set(consulta_tokens):
            if w not in self.idf: continue
            for i, c in enumerate(self.tf):
                f = c.get(w, 0)
                if f: sc[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return [self.ids[i] for i in np.argsort(-sc)[:k]]
print(len(train_texts), "docs de entrenamiento;", {l: len(v) for l, v in test_by_lang.items()}, "docs de test")

## Paso 1 · Entrenar los tres tokenizers con el mismo pipeline

**TODO.** Implementa `entrena_tokenizer` (es la función `construye_hf` de la lección, §7.1, generalizada). Requisitos: normalizer `NFKC`; pre-tokenizer = `Metaspace(replacement="▁", prepend_scheme="always")` seguido de `Punctuation(behavior="isolated")`; modelo y *trainer* según `tipo`; `[UNK]` como único token especial; entrena con `train_from_iterator(textos, trainer)`.

<details><summary>🪜 Pista 1 (idea)</summary>

Solo cambian dos cosas entre los tres tipos: el objeto `models.*` y el `trainers.*`. El resto del pipeline es idéntico (por eso la comparación es justa).
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>

`models.BPE(unk_token="[UNK]")` + `trainers.BpeTrainer(vocab_size=..., special_tokens=["[UNK]"])`; `models.WordPiece(unk_token="[UNK]")` + `trainers.WordPieceTrainer(..., continuing_subword_prefix="##")`; `models.Unigram()` + `trainers.UnigramTrainer(..., unk_token="[UNK]")`. Asigna `tok.normalizer = normalizers.NFKC()` y `tok.pre_tokenizer = pre_tokenizers.Sequence([...])` (los dos pre-tokenizers en ese orden) antes de entrenar.
</details>

In [ ]:
def entrena_tokenizer(tipo: str, vocab_size: int, textos: list) -> Tokenizer:
    """Devuelve un Tokenizer HF entrenado (tipo: 'bpe' | 'wordpiece' | 'unigram')."""
    # TODO: modelo + trainer según `tipo`; normalizer NFKC; pre_tokenizer Metaspace + Punctuation; entrenar
    raise NotImplementedError("TODO paso 1")

def intenta(f, *a, **k):
    """Ejecuta un paso; si aún tiene TODO pendiente avisa sin romper 'Ejecutar todo'."""
    try: return f(*a, **k)
    except NotImplementedError as e: print("⏳ pendiente:", e); return None

toks = {t: intenta(entrena_tokenizer, t, VOCAB, train_texts) for t in ["bpe", "wordpiece", "unigram"]}
print({t: (v.get_vocab_size() if v else None) for t, v in toks.items()})

## Paso 2 · Evaluar: fertilidad, bytes/token, `[UNK]` de estrés y morfología

**TODO.** `evalua(tok, nombre)` devuelve un dict con: `fert_es/en/code`, `bpt_es/en/code` (bytes por token, usa `metricas_tokenizacion`), `unk_estres` (suma de `n_unk` sobre `ESTRES`) y `f1_morf_es/en` (usa `f1_morfologico` con `lambda w: tok.encode(w).tokens`).

<details><summary>🪜 Pista 1</summary>

`metricas_tokenizacion(encode, textos)` ya devuelve `fertilidad` y `bytes_por_token`; `encode` es `lambda s: tok.encode(s).tokens`.
</details>
<details><summary>🪜 Pista 2</summary>

Bucle `for lang in ["es","en","code"]`: `m = metricas_tokenizacion(..., test_by_lang[lang])`; `out[f"fert_{lang}"] = m["fertilidad"]`, `out[f"bpt_{lang}"] = m["bytes_por_token"]`.
</details>

In [ ]:
def evalua(tok: Tokenizer, nombre: str) -> dict:
    # TODO: métricas por idioma, [UNK] de estrés y F1 morfológica
    raise NotImplementedError("TODO paso 2")

filas = [intenta(evalua, tok, t) for t, tok in toks.items() if tok is not None]
filas = [f for f in filas if f]
if filas: print(pd.DataFrame(filas).set_index("nombre").round(3).T)

## Paso 3 · ¿Cambia el tokenizer la recuperación? BM25 con cada analizador

**TODO.** Construye un índice BM25 por tokenizer sobre **todos los documentos** del corpus y calcula el nDCG@10 medio en las consultas del mini golden set, **degradadas** (se descarta ~40 % de las palabras de cada consulta, semilla fija) para que no sea trivial. El analizador es `lambda t: tok.encode(t.lower()).tokens`.

<details><summary>🪜 Pista 1</summary>

`BM25(textos, ids, analizador)` ya está hecho; llama a `.ranking(tokens_de_la_consulta, k=10)` y usa `ndcg_at_k(ranking, corpus["qrels"][qid], 10)`.
</details>
<details><summary>🪜 Pista 2</summary>

Degradar: `rr = random.Random(SEED); q = " ".join(w for w in texto.split() if rr.random() > 0.4) or texto`.
</details>

In [ ]:
def ndcg_bm25(tok: Tokenizer) -> float:
    # TODO: índice BM25 con analizador del tokenizer; nDCG@10 medio sobre corpus["queries"] degradadas
    raise NotImplementedError("TODO paso 3")

ndcg = {t: intenta(ndcg_bm25, tok) for t, tok in toks.items() if tok is not None}
print(ndcg)

## Paso 4 · Exportar un BPE byte-level a `tiktoken`

**TODO.** Entrena un BPE byte-level (regex GPT-2 + `ByteLevel`, `vocab_size=VOCAB`) y conviértelo a `tiktoken.Encoding`; `export_check(...)` devuelve el **nº de textos de test con tokenización distinta** (debe ser 0). Te damos `bytes_a_unicode`; la conversión de la lección (`hf_a_tiktoken`) es tu referencia.

<details><summary>🪜 Pista 1</summary>

Invierte `bytes_a_unicode` para pasar de los tokens HF (alfabeto visible) a `bytes`; reserva los ranks 0–255 para los bytes sueltos y numera los merges en orden de id.
</details>
<details><summary>🪜 Pista 2</summary>

`ranks = {bytes([i]): i for i in range(256)}`; por cada `(id, bytes)` ordenado con `len(b) > 1`: `ranks[b] = len(ranks)`; `tiktoken.Encoding(name=..., pat_str=PAT_GPT2, mergeable_ranks=ranks, special_tokens={"<|endoftext|>": len(ranks)})`.
</details>

In [ ]:
import tiktoken
PAT_GPT2 = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
def bytes_a_unicode() -> dict:
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))

def export_check(textos: list, vocab_size: int) -> int:
    # TODO: entrenar BPE byte-level, convertirlo a tiktoken.Encoding y contar textos con tokenización distinta
    raise NotImplementedError("TODO paso 4")

n_dif = intenta(export_check, [t for ts in test_by_lang.values() for t in ts], VOCAB)
print("textos distintos tras exportar:", n_dif)

## Paso 5 · La decisión

**TODO.** Asigna a `DECISION` una de `"bpe"`, `"wordpiece"`, `"unigram"` y a `JUSTIFICACION` un texto (≥ 300 caracteres) que cite **al menos tres cifras** de tu tabla (fertilidad, bytes/token, `[UNK]`, F1 morfológica, nDCG@10) **y un riesgo** de tu elección. Piensa en el índice sparse de Políglota (ES/EN/código) y en que más adelante entrenarás un embedder sobre este vocabulario.

<details><summary>🪜 Pista 1 (estructura)</summary>

Cuatro frases: (1) la elección; (2) la métrica que más pesa para Políglota y su cifra frente a las otras dos familias; (3) dos cifras más que la apoyan o la matizan (p. ej. `[UNK]` en estrés y nDCG@10); (4) el riesgo y cómo lo mitigarías.
</details>
<details><summary>🪜 Pista 2 (criterios)</summary>

Para un índice sparse importan la F1 morfológica (fronteras ↔ términos con sentido) y el nDCG@10; para el embedder futuro, la fertilidad (longitud de secuencia = coste) y la ausencia de `[UNK]`. Si el nDCG@10 empata entre familias, la decisión la deciden las otras métricas: dilo explícitamente (repasa la tabla de §9 de la lección).
</details>

In [ ]:
DECISION = None        # TODO: "bpe" | "wordpiece" | "unigram"
JUSTIFICACION = ""     # TODO: ≥ 300 caracteres, ≥ 3 cifras de tu tabla, 1 riesgo

## Rúbrica (autoevaluación)

Los umbrales absolutos están calibrados con la **solución de referencia** (corpus espejo Europarl+stdlib, `FAST_DEV_RUN=True`, `VOCAB=4000`): `fert_es` BPE ≈ 1.46, Unigram ≈ 1.62 (umbral ≤ 1.80); bytes/token ES de BPE ≈ 3.9 (umbral ≥ 3.3); F1 morfológica ES BPE ≈ 0.29, Unigram ≈ 0.42 (umbral ≥ 0.33); nDCG@10 BM25 ≈ 0.95 con los tres (umbral 0.85); exportación a tiktoken = 0 diferencias.

In [ ]:
def rubrica():
    """Puntúa los entregables. Umbrales absolutos calibrados con la solución de referencia (menos margen); los relativos son los robustos."""
    R = []
    ok = lambda nombre, cond, pts: R.append((nombre, bool(cond), pts))
    F = {f["nombre"]: f for f in filas} if filas else {}
    ok("1 · los 3 tokenizers entrenan y llegan a ≥ 95 % del vocabulario pedido", all(v is not None and v.get_vocab_size() >= 0.95 * VOCAB for v in toks.values()), 2)
    if len(F) == 3:
        ok("2 · fertilidad ES de BPE ≤ 1.80 y de WordPiece ≤ 1.80", F["bpe"]["fert_es"] <= UMBRALES["fert_es_max"] and F["wordpiece"]["fert_es"] <= UMBRALES["fert_es_max"], 2)
        ok("2 · el código necesita más tokens/palabra que el español (fert_code > fert_es) en los 3", all(f["fert_code"] > f["fert_es"] for f in F.values()), 1)
        ok("2 · bytes/token ES de BPE ≥ " + str(UMBRALES["bpt_es_min"]), F["bpe"]["bpt_es"] >= UMBRALES["bpt_es_min"], 1)
        ok("2 · Unigram alinea mejor la morfología ES que BPE (F1 relativa)", F["unigram"]["f1_morf_es"] > F["bpe"]["f1_morf_es"], 2)
        ok("2 · F1 morfológica ES de Unigram ≥ " + str(UMBRALES["f1_es_min"]), F["unigram"]["f1_morf_es"] >= UMBRALES["f1_es_min"], 1)
        ok("2 · WordPiece con un solo [UNK] por palabra: unk_estres(WP) ≤ unk_estres(BPE)", F["wordpiece"]["unk_estres"] <= F["bpe"]["unk_estres"], 1)
    else:
        ok("2 · evalúa los 3 tokenizers", False, 8)
    nd = {k: v for k, v in ndcg.items() if v is not None}
    ok("3 · nDCG@10 BM25 ≥ " + str(UMBRALES["ndcg_min"]) + " con los 3 analizadores", len(nd) == 3 and min(nd.values()) >= UMBRALES["ndcg_min"], 2)
    ok("4 · exportación a tiktoken con 0 diferencias", n_dif == 0, 2)
    ok("5 · decisión válida", DECISION in ("bpe", "wordpiece", "unigram"), 1)
    ok("5 · justificación ≥ 300 caracteres con ≥ 3 cifras", len(JUSTIFICACION) >= 300 and len(re.findall(r"\d+[.,]?\d*", JUSTIFICACION)) >= 3, 2)
    tot = sum(p for _, _, p in R); got = sum(p for _, c, p in R if c)
    for nombre, c, p in R: print(("✅" if c else "⬜"), f"[{p}]", nombre)
    print(f"\nPuntuación: {got}/{tot}")
UMBRALES = {"fert_es_max": 1.80, "bpt_es_min": 3.3, "f1_es_min": 0.33, "ndcg_min": 0.85}
rubrica()

## ⛔ SPOILER — intenta resolverlo primero

Debajo está la **solución de referencia completa**. Si llegas aquí sin haber intentado los pasos 1–5, vuelve arriba: el valor está en equivocarte con el `[UNK]` de Unigram o con el prefijo `##`.
Al ejecutar las celdas de solución, las funciones se redefinen y la rúbrica final te muestra la puntuación de la referencia.

### Solución · pasos 1 y 2

In [ ]:
def entrena_tokenizer(tipo: str, vocab_size: int, textos: list) -> Tokenizer:
    if tipo == "bpe":
        tok, tr = Tokenizer(models.BPE(unk_token="[UNK]")), trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    elif tipo == "wordpiece":
        tok, tr = Tokenizer(models.WordPiece(unk_token="[UNK]")), trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"], continuing_subword_prefix="##")
    elif tipo == "unigram":
        tok, tr = Tokenizer(models.Unigram()), trainers.UnigramTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"], unk_token="[UNK]")
    else:
        raise ValueError(tipo)
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="always"), pre_tokenizers.Punctuation(behavior="isolated")])
    tok.train_from_iterator(textos, tr)
    return tok

def evalua(tok: Tokenizer, nombre: str) -> dict:
    out = {"nombre": nombre}
    for lang in ["es", "en", "code"]:
        m = metricas_tokenizacion(lambda s: tok.encode(s).tokens, test_by_lang[lang])
        out[f"fert_{lang}"], out[f"bpt_{lang}"] = m["fertilidad"], m["bytes_por_token"]
    out["unk_estres"] = sum(n_unk(tok, t) for t in ESTRES)
    out["f1_morf_es"] = f1_morfologico(lambda w: tok.encode(w).tokens, MORF_ES)
    out["f1_morf_en"] = f1_morfologico(lambda w: tok.encode(w).tokens, MORF_EN)
    return out

toks = {t: entrena_tokenizer(t, VOCAB, train_texts) for t in ["bpe", "wordpiece", "unigram"]}
filas = [evalua(tok, t) for t, tok in toks.items()]
print(pd.DataFrame(filas).set_index("nombre").round(3).T)

### Solución · pasos 3 y 4

In [ ]:
ids_docs = [d["id"] for d in docs]; textos_docs = [d["text"] for d in docs]

def ndcg_bm25(tok: Tokenizer) -> float:
    bm = BM25(textos_docs, ids_docs, lambda t: tok.encode(t.lower()).tokens)
    rr = random.Random(SEED); vals = []
    for q in corpus["queries"]:
        texto = " ".join(w for w in q["text"].split() if rr.random() > 0.4) or q["text"]
        vals.append(ndcg_at_k(bm.ranking(tok.encode(texto.lower()).tokens, 10), corpus["qrels"][q["qid"]], 10))
    return float(np.mean(vals))

ndcg = {t: ndcg_bm25(tok) for t, tok in toks.items()}
print({t: round(v, 3) for t, v in ndcg.items()})

def export_check(textos: list, vocab_size: int) -> int:
    bl = Tokenizer(models.BPE())
    bl.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(PAT_GPT2), behavior="isolated"), ByteLevel(add_prefix_space=False, use_regex=False)])
    bl.decoder = decoders.ByteLevel()
    bl.train_from_iterator(train_texts, trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=ByteLevel.alphabet(), special_tokens=["<|endoftext|>"]))
    u2b = {v: k for k, v in bytes_a_unicode().items()}
    piezas = sorted((i, bytes(u2b[ch] for ch in t)) for t, i in bl.get_vocab().items() if t != "<|endoftext|>")
    ranks = {bytes([i]): i for i in range(256)}
    for _, b in piezas:
        if len(b) > 1: ranks[b] = len(ranks)
    enc = tiktoken.Encoding(name="poliglota_proyecto", pat_str=PAT_GPT2, mergeable_ranks=ranks, special_tokens={"<|endoftext|>": len(ranks)})
    dif = 0
    for t in textos:
        a = [bytes(u2b[ch] for ch in x) for x in bl.encode(t).tokens]
        dif += a != [enc.decode_single_token_bytes(i) for i in enc.encode_ordinary(t)]
    return dif

n_dif = export_check([t for ts in test_by_lang.values() for t in ts], VOCAB)
print("textos distintos tras exportar:", n_dif)

### Solución · paso 5 y rúbrica final

In [ ]:
T = pd.DataFrame(filas).set_index("nombre")
DECISION = "unigram" if T.loc["unigram", "f1_morf_es"] > T.loc["bpe", "f1_morf_es"] + 0.05 else "bpe"
JUSTIFICACION = (
    f"Recomiendo {DECISION}. Con |V|={VOCAB} la fertilidad en español es {T.loc['bpe','fert_es']:.2f} (BPE) frente a {T.loc['unigram','fert_es']:.2f} (Unigram), es decir Unigram necesita "
    f"≈ {T.loc['unigram','fert_es']/T.loc['bpe','fert_es']-1:.0%} más tokens, pero su F1 morfológica en español es {T.loc['unigram','f1_morf_es']:.2f} frente a {T.loc['bpe','f1_morf_es']:.2f}, "
    f"lo que favorece el índice sparse con raíces/sufijos reales; el nDCG@10 de BM25 fue {ndcg['unigram']:.2f} (Unigram) y {ndcg['bpe']:.2f} (BPE). "
    f"Riesgo: ninguno de los tres es byte-level, así que los caracteres no vistos producen [UNK] ({T.loc['unigram','unk_estres']} en el texto de estrés); "
    f"antes de entrenar el embedder propio habrá que añadir byte fallback y medir paridad multilingüe (módulo 04)."
)
print(DECISION, "|", len(JUSTIFICACION), "caracteres")
rubrica()

## Retos extra (nivel experto)

1. **Controla el tamaño.** Repite la comparación con SentencePiece (`unigram` y `bpe`, `byte_fallback=True`) a `vocab_size` idéntico y añade al informe el *round-trip* (`decode(encode(t)) == t`) con la configuración por defecto y con `identity`.
2. **Barrido de vocabulario.** Entrena BPE/Unigram a 1 k, 2 k, 4 k, 8 k y 16 k (`FAST_DEV_RUN=False`) y dibuja fertilidad y F1 morfológica; ¿dónde se aplana la curva?
3. **Tokenizer para sparse retrieval.** Compara el nDCG@10 de BM25 con el tokenizer *de palabras* (módulo 00) frente a Unigram y BPE usando consultas con inflexiones distintas (singular/plural); ¿quién gana y por qué?
4. **BPE-dropout como augmentación de consultas.** Genera consultas con `bpe_pieza(..., dropout=0.1)` y mide el cambio de nDCG@10.
5. **Implementa el criterio de verosimilitud** de la lección dentro de un entrenador a escala del corpus completo (¡cuida la complejidad!) y compara su vocabulario con el de `BpeTrainer`.

## Reflexión (conecta con producción y con Políglota)

- ¿Qué cambiaría en tu decisión si el 40 % de los documentos fueran código? ¿Y si el buscador tuviera que indexar nombres propios y códigos de error?
- Un tokenizer **se congela** al entrenar el embedder (módulo 09–11): ¿qué coste tiene cambiarlo después (re-indexar, re-entrenar embeddings)? ¿Qué harías en la v1 para no arrepentirte?
- ¿Cómo detectarías en producción que el vocabulario ya no encaja con los datos (drift de dominio/idioma)? (Pista: módulo 05.)